## Loading Dataset

In [6]:
def load_conll(file_path):
    sentences = []
    sentence = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip() == "" or line.startswith("-DOCSTART-"):
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            else:
                parts = line.strip().split()
                word = parts[0]
                tag = parts[-1]   # NER tag
                sentence.append((word, tag))
    
    return sentences

file_path = "C:\\Users\\malai\\conll2003\\eng.train"

data = load_conll(file_path)



In [7]:
print("Total sentences:", len(data))
print("First sentence:", data[0])


Total sentences: 14041
First sentence: [('EU', 'B-ORG'), ('rejects', 'O'), ('German', 'B-MISC'), ('call', 'O'), ('to', 'O'), ('boycott', 'O'), ('British', 'B-MISC'), ('lamb', 'O'), ('.', 'O')]


## Convert into plain sentences

In [8]:
sentences = []

for sent in data:
    words = [word for word, tag in sent]
    sentence = " ".join(words)
    sentences.append(sentence)

print("Example sentence:", sentences[0])


Example sentence: EU rejects German call to boycott British lamb .


## Text Preprocessing

In [19]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')


import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Lowercase
    text = text.lower()
    
    # Tokenize
    words = word_tokenize(text)
    
    # Remove punctuation + stopwords
    words = [w for w in words if w.isalnum() and w not in stop_words]
    
    return " ".join(words)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\malai\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\malai\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


## Create queries automatically

In [20]:
queries = []
intents = []

for sentence in sentences[:1000]:   # limit to 500 for now (faster)
    
    queries.append("Find names in: " + sentence)
    intents.append("find_person")
    
    queries.append("Find locations in: " + sentence)
    intents.append("find_location")
    
    queries.append("Find organizations in: " + sentence)
    intents.append("find_organization")
    
    queries.append("Find dates in: " + sentence)
    intents.append("find_date")


In [21]:
print(queries[0])
print(intents[0])

print("Total queries:", len(queries))


Find names in: EU rejects German call to boycott British lamb .
find_person
Total queries: 4000


In [22]:
processed_queries = [preprocess(q) for q in queries]


## Convert text  into TF-IDF features

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [24]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(processed_queries)
y = intents


In [25]:
print(X.shape)   # rows, features


(4000, 2817)


## Train Intent Classification Model

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

sample = ["Find names in: Ali went to Lahore"]
sample_vec = vectorizer.transform(sample)

prediction = model.predict(sample_vec)

print("Predicted Intent:", prediction[0])


Predicted Intent: find_person


## Model Evaluation

In [27]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print(classification_report(y_test, y_pred))


Accuracy: 0.65125
                   precision    recall  f1-score   support

        find_date       0.57      0.71      0.63       185
    find_location       0.71      0.64      0.67       197
find_organization       0.74      0.64      0.68       217
      find_person       0.61      0.63      0.62       201

         accuracy                           0.65       800
        macro avg       0.66      0.65      0.65       800
     weighted avg       0.66      0.65      0.65       800



## Connect Intent + NER + POS Tagging

In [32]:
from nltk import pos_tag, word_tokenize

nltk.download('averaged_perceptron_tagger_eng')

ignore_words = ["find", "in", "names", "locations", "organizations", "date"]
locations_list = ["Lahore", "Karachi", "Islamabad", "London", "Paris"]


def extract_entities_from_text(text):
    words = word_tokenize(text)
    tagged = pos_tag(words)

    entities = {
        "Person": [],
        "Location": [],
        "Organization": []
    }

    for word, tag in tagged:
        word_lower = word.lower()
        
        # Skip unwanted words
        if word_lower in ignore_words:
            continue
        
        # Location check
        if word in locations_list:
            entities["Location"].append(word)
        
        # Person check (proper noun)
        elif tag == "NNP":
            entities["Person"].append(word)

    return entities



def chatbot(user_input):
    
    # Step 1: Predict intent
    processed = preprocess(user_input)
    vec = vectorizer.transform([processed])
    intent = model.predict(vec)[0]
    
    # Step 2: Extract entities directly from user input
    entities = extract_entities_from_text(user_input)
    
    # Step 3: Generate response
    if intent == "find_person":
        if entities["Person"]:
            return "Persons: " + ", ".join(entities["Person"])
        else:
            return "No person found"
    
    elif intent == "find_location":
        if entities["Location"]:
            return "Locations: " + ", ".join(entities["Location"])
        else:
            return "No location found"
    
    elif intent == "find_organization":
        if entities["Organization"]:
            return "Organizations: " + ", ".join(entities["Organization"])
        else:
            return "No organization found"
    
    else:
        return "Intent not recognized"


print(chatbot("Find names in: Ali went to Lahore"))




Persons: Ali


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\malai\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [34]:
import pickle

# Save model
with open("intent_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)
